In [1]:
%%configure -f
{
    "conf": {
        "spark.jars": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar ,s3://ne-prod-delta-lake-secure-layer/elasticmapreduce/external-jars/encrypt-decrypt-spark-1.0-SNAPSHOT.jar",
        "spark.submit.pyFiles": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar,s3://ne-prod-data-pipeline-orchestrator/ds-da-repo/emr_spark_utils.zip"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1781759717505_0001,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_adhikansh_goel_slicebank_com,
5,application_1781759717505_0006,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_sinjini_roy_slicebank_com,
6,application_1781759717505_0007,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_adhikansh_goel_slicebank_com,


In [2]:
from decimal import Decimal
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import openpyxl as xl
from openpyxl.utils import get_column_interval
from openpyxl.utils.dataframe import dataframe_to_rows
import re
import json
import operator
import boto3
import email
import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import time
import enum
from pyspark.sql import SparkSession
import math
import io
from pyspark.sql import functions as F
import pytz
import requests

from pyspark.sql.functions import col, split, get_json_object, date_sub, current_timestamp, max as spark_max, min as spark_min, collect_list, slice
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import col, split, get_json_object, current_date, date_sub, year, month, dayofmonth, broadcast, expr, sort_array, struct
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import explode, row_number

from pyspark.sql.window import Window
# Compute yesterday and today
yesterday = date_sub(current_date(), 1)
today = current_date()


s3 = boto3.client('s3')
date_yesterday = datetime.now() - timedelta(days=1)

date_yesterday_str = date_yesterday.strftime('%Y-%m-%d')

def get_secrets(secret_name):
    region_name = "ap-south-1"
    session = boto3.session.Session()

    client = session.client(service_name="secretsmanager", region_name=region_name)

    try:
        get_secret_value_response = client.get_secret_value(SecretId=secret_name)

    except ClientError as e:
        if e.response.get("Error").get("Code") == "ResourceNotFoundException":
            raise Exception("The requested secret " + secret_name + " was not found")
        elif e.response.get("Error").get("Code") == "InvalidRequestException":
            raise Exception("The request was invalid due to:", e)
        else:
            raise Exception("The request failed because of:", e)

    secret = get_secret_value_response.get("SecretString")

    if isinstance(secret, str):
        secret = eval(secret)

    return secret

MSSQL_ANALYTICS_CLUSTER_COMMON = "prod/data/analytics/cbs/common" #verified
BANKOS = "prod/data/analytics/bankos"
MIS_8004 = "prod/data/analytics/cbs/mis_8004"
MIS_2397 = "prod/data/analytics/cbs/mis_2397"
BANK_KYC = "prod/data/analytics/bank_kyc"
BANK_KYC_RW = "prod/data/analytics/bankkyc_rw"
BANK_ONBOARDING_DB = "prod/data/analytics/bank_onboarding"
BANK_ONBOARDING_DB_RW = "prod/data/analytics/bank_onboarding_rw"
COMMON_TOKEN = 'prod/data/analytics/common-token'
SUNRISE_MAIL = 'prod/data/analytics/sunrise-mail'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V2  = 'prod/data/analytics/bank-payment-links-access-token-v2'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V3  = 'prod/data/analytics/bank-payment-links-access-token-v3'
RAZORPAY = 'prod/data/analytics/cbs/razorpay'
HELO_AI = 'prod/data/analytics/heloai'
SERVICE_TOKEN = 'prod/data/analytics/service-token'
CONFLUENT_TOKEN = 'prod/data/analytics/confluent-token'

    
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
from datetime import datetime
slack_token =get_secrets(COMMON_TOKEN)["slack_token"]
client = WebClient(token=slack_token)

spark.udf.registerJavaFunction("decryptFunction", "com.piixie.SparkDecrypt")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

def push_to_slack(df1):
    channel_id = "C0958LSLM8Q"
    df1.to_csv('./sample.csv', index=False)
    if df1.shape[0] > 0 : 
        client.files_upload_v2(
        channel = channel_id,
        title = "df_base",
        file = "./sample.csv",
        initial_comment="df_base",
        )
        print(f'pushed to slack: {channel_id}')

def load_from_s3(path, view_name):
    data = spark.read.parquet(path)
    data.createOrReplaceTempView(view_name)
    
    return data
    
def load_csv(path, view_name):    
    import pandas as pd
    import io
    import boto3

    # Parse S3 path
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    data = io.BytesIO(obj['Body'].read())
    df = pd.read_csv(data)

    # Convert to Spark DataFrame
    df = spark.createDataFrame(df)

    # Create temporary view
    df.createOrReplaceTempView(view_name)

def load_excel(path, view_name):
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    excel_data = io.BytesIO(obj['Body'].read())
    df = pd.read_excel(excel_data)
    df.columns = df.columns.str.strip()

    # Convert to Spark DataFrame
    blocked_df = spark.createDataFrame(df)
    blocked_df.createOrReplaceTempView(view_name)
    
    return blocked_df

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

######### TOKEN SET RATIO

from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import IntegerType
import re
from difflib import SequenceMatcher

def tokenize(s):
    if s is None:
        return []
    s = re.sub(r'[^a-zA-Z0-9 ]', '', s.lower())
    return s.split()

def ratio(s1, s2):
    return int(SequenceMatcher(None, s1, s2).ratio() * 100)

def token_set_ratio(s1, s2):
    if s1 is None or s2 is None:
        return 0
    
    tokens1 = set(tokenize(s1))
    tokens2 = set(tokenize(s2))
    
    common_tokens = tokens1.intersection(tokens2)
    diff1 = tokens1.difference(tokens2)
    diff2 = tokens2.difference(tokens1)
    
    sorted_common = ' '.join(sorted(common_tokens))
    sorted_diff1 = ' '.join(sorted(diff1))
    sorted_diff2 = ' '.join(sorted(diff2))
    
    # Combine strings in three ways and calculate ratio
    combined1 = sorted_common + ' ' + sorted_diff1
    combined2 = sorted_common + ' ' + sorted_diff2
    
    ratios = [
        ratio(sorted_common, combined1),
        ratio(sorted_common, combined2),
        ratio(combined1, combined2),
    ]
    
    return max(ratios)


token_set_ratio_udf = udf(token_set_ratio, IntegerType())

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
7,application_1781759717505_0008,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_rayansh_khamesra_slicebank_com,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
spark.sql("""
SELECT 
    user_uuid, 
    src_txn_id,
    narration,
    debit_party_type,
    credit_party_type
FROM casa_txn_gold.transaction
WHERE year = 2026
""").createOrReplaceTempView('casa_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
df = spark.sql("""
SELECT 
    a.*, 
    b.account_open_date as onboarding_date,
    c.narration as casa_narration,
    c.debit_party_type,
    c.credit_party_type,
    d.note as upi_note
FROM cyber_crime_gold.final_view_deduped a
JOIN bsgcore_gold.account_master b on a.account_id = b.account_id and b.is_active = 1 and b.__is_deleted=False and b.product_code = 1150
LEFT JOIN casa_data c on a.src_txn_id = c.src_txn_id
LEFT JOIN upiswitch_tpap_gold.cbs_transactions d on d.year = 2026 and a.rrn = d.rrn
WHERE a.user_account_number = a.account_no
""").createOrReplaceTempView('dca_txn_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:


dca_txn_df = spark.sql("""
SELECT a.*, b.dsa_account_holder_name as user_name
FROM dca_txn_data a
JOIN s3_tool_propagator_pii.dca_fraud_base_data b on a.uuid = b.uuid
where b.kyc_state in (
'Bihar', 'Assam', 'Punjab', 'Rajasthan', 'Maharashtra', 'Jharkhand', 'Chandigarh'
)
and cast(b.onboarding_date as date) between date '2026-06-01' and date '2026-06-18'
""")

dca_txn_df = dca_txn_df.withColumn("cp_name_match", token_set_ratio_udf(col("user_name"), col("counter_party_cbs_name")))

dca_txn_df.createOrReplaceTempView('dca_txn_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
spark.sql("""
SELECT
    *,
    CASE
        WHEN REGEXP_LIKE(upi_note, '^[0-9]+$') THEN 'ALL_NUMERIC'
        WHEN REGEXP_LIKE(upi_note, '^[A-Za-z]+$') THEN 'ALL_ALPHA'
        WHEN REGEXP_LIKE(upi_note, '^[A-Za-z0-9]+$') THEN 'ALPHANUMERIC'
        ELSE 'OTHER'
    END AS note_pattern,
    LENGTH(upi_note) as note_length
FROM dca_txn_data
WHERE txn_mode <> 'IFT - Interest Credits' 
      AND NOT (
          LOWER(upi_note) IN (
                'money', 'game', 'shop', 'test', 'hello', 'book', 'food', 'milk',
                'lunch', 'testing', 'rent', 'curd', 'tshirts', 'shirt', 'library',
                'bill', 'cash',
                'general', 'coffee', 'rice', 'fruits', 'juice', 'water', 'banana',
                'paneer', 'bread', 'biscuit', 'dinner', 'fruit', 'sweets', 'bakery',
                'pizza', 'cake', 'sweet', 'beer', 'market', 'samosa', 'fish', 'fuel',
                'pork', 'lassi', 'onion', 'tomato', 'meat', 'beef', 'mutton', 'wine',
                'masala', 'bazar', 'bajar', 'chana', 'potato', 'salt', 'butter',
                'eggs', 'honey', 'sabun', 'sabji', 'sabje', 'dalia', 'momo', 'puri',
                'mithai', 'allu', 'apple', 'ciggy', 'lays','chicken',
                'petrol', 'auto', 'express', 'rapido', 'parking',
                'xerox', 'dress', 'mandate', 'mobile', 'advance', 'goods', 'hotel',
                'good', 'brush', 'shoes', 'shoe', 'parts', 'blouse', 'print', 'printer',
                'flower', 'medical', 'drinks', 'misc', 'clear', 'diaper', 'lighter',
                'gold', 'tube', 'laptop', 'change', 'canteen', 'cement', 'battery',
                'charger', 'tyre', 'duck', 'surf', 'comfort', 'shampoo', 'grocery',
                'bricks', 'screw', 'sand', 'cartons',
                'payout', 'paid', 'send', 'salary', 'lottery',
                'kitchen', 'light', 'lucky', 'remarks', 'stellar',
                'meet', 'babe', 'babu'
          )
          OR LOWER(upi_note) LIKE 'payment from%'
          OR LOWER(upi_note) LIKE 'paid via%'
          OR LOWER(upi_note) LIKE 'sent using%'
          OR LOWER(upi_note) LIKE 'sent from%'
          OR LOWER(upi_note) LIKE 'sent via%'
          OR LOWER(upi_note) LIKE 'payment%'
          OR LOWER(upi_note) LIKE 'cloth%'
      )
""").createOrReplaceTempView('dca_analysis_data')



VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
%%pretty
df = spark.sql("""WITH running_totals AS (

    SELECT
        account_no,
        uuid,
        txn_ref_no,
        transactiontime,
        txn_amount,
        note_length,
        upi_note,

        SUM(
            CASE WHEN note_length IN (4,5,6)
            THEN txn_amount ELSE 0 END
        ) OVER (
            PARTITION BY account_no
            ORDER BY transactiontime
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_amount,

        SUM(
            CASE WHEN note_length IN (4,5,6)
            THEN 1 ELSE 0 END
        ) OVER (
            PARTITION BY account_no
            ORDER BY transactiontime
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_count

    FROM dca_analysis_data
    WHERE txn_nature = 'C'

),

anchor_txn AS (

    SELECT *
    FROM (

        SELECT
            account_no,
            transactiontime AS anchor_time,

            ROW_NUMBER() OVER (
                PARTITION BY account_no
                ORDER BY transactiontime
            ) rn

        FROM running_totals
        WHERE running_amount > 40000
          AND running_count > 25

    ) x
    WHERE rn = 1

)

SELECT
    r.account_no,
    r.uuid,
    r.txn_ref_no,
    r.transactiontime,
    r.txn_amount,
    r.running_amount,
    r.running_count,
    r.upi_note
FROM running_totals r
JOIN anchor_txn a
    ON r.account_no = a.account_no

WHERE r.note_length IN (4,5,6)
  AND r.transactiontime <= a.anchor_time

ORDER BY
    r.account_no,
    r.transactiontime""").show(200,0)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

account_no,uuid,txn_ref_no,transactiontime,txn_amount,running_amount,running_count,upi_note
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060233072201,2026-06-02 07:55:19.96,200.0,200.0,1,0004O
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060233087101,2026-06-02 07:55:33.703,200.0,400.0,2,0004P
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060237333001,2026-06-02 08:16:53.823,300.0,700.0,3,0004S
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060241889101,2026-06-02 08:36:47.43,250.0,950.0,4,0004T
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060247077301,2026-06-02 08:56:55.48,300.0,1250.0,5,0004U
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060249182001,2026-06-02 09:04:28.81,200.0,1450.0,6,0002B
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060249213201,2026-06-02 09:04:38.59,200.0,1650.0,7,0002C
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060249242801,2026-06-02 09:04:42.853,200.0,1850.0,8,0002E
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060249537101,2026-06-02 09:05:45.233,300.0,2150.0,9,0002D
033311501073573,169117e0-d44b-4bb2-ae4d-658445851fd2,2026060249964101,2026-06-02 09:07:14.44,200.0,2350.0,10,0002G
